In [45]:
import datasets as ds 
import pandas as pd 
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

In [2]:
df = pd.read_csv('dataset/TMDB_IMDB_Movies_Dataset.csv')

In [3]:
df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'tconst', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords', 'directors', 'writers', 'averageRating', 'numVotes',
       'cast'],
      dtype='str')

In [4]:
# select only untrain xgboost dataset
df = df[(df['revenue'] <= 10000) & (df['budget'] <= 10000)]
df.shape

(407602, 29)

In [5]:
# df['cleaned_genres'] = df['genres'].dropna().apply(lambda x: [i.strip() for i in x.split(',') if i.strip()])
df['cleaned_genres'] = df['genres'].dropna().apply(lambda x: [i for i in x.split(', ')])
df['cleaned_genres'].head(n=5)

395                   [Comedy, Romance]
509                   [Romance, Comedy]
872                   [Comedy, Romance]
944    [Crime, Drama, Thriller, Action]
975                  [Horror, Thriller]
Name: cleaned_genres, dtype: object

In [6]:
genres_dummies = df['genres'].astype(str).str.get_dummies(sep=', ')
top_genres = genres_dummies.sum().sort_values(ascending=False).head(12).index
print(top_genres)

Index(['Drama', 'Comedy', 'Documentary', 'Romance', 'Thriller', 'Action',
       'Horror', 'Animation', 'Crime', 'TV Movie', 'Family', 'Music'],
      dtype='str')


In [7]:
df['top_genres'] = df['cleaned_genres'].dropna().apply(lambda x: [genre for genre in x if genre in top_genres])
df = df.dropna(subset=['top_genres', 'overview'], how='any')
df = df[df['overview'].str.strip() != ""]
df[df['top_genres'] != df['cleaned_genres']].head(n=5)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast,cleaned_genres,top_genres
1033,9732,The Lion King II: Simba's Pride,6.941,4130,Released,1998-10-24,0,81,False,/vyOFlV1afDHb9MMwJVyON4qXbvC.jpg,...,United States of America,English,"friendship, baby, africa, lion, shakespeare in...","Darrell Rooney, Rob LaDuca","Flip Kobler, Cindy Marcus, Jenny Wingfield, Li...",6.4,82528,"Matthew Broderick, Neve Campbell, Jason Marsde...","[Family, Adventure, Animation, Action]","[Family, Animation, Action]"
1179,8965,Atlantis: Milo's Return,6.334,3728,Released,2003-02-25,0,80,False,/GDTwA5FBvVUOjCGKoNDMflbs3Z.jpg,...,United States of America,English,"submarine, atlantis, cartoon, animal attack, v...","Victor Cook, Toby Shelton, Tad Stones","Thomas Hart, Henry Gilroy, Kevin Hopps, Tad St...",5.0,12683,"James Arnold Taylor, Cree Summer, John Mahoney...","[Fantasy, Animation, Science Fiction, Family, ...","[Animation, Family, Action]"
1197,537996,The Ballad of Buster Scruggs,7.145,3684,Released,2018-11-09,0,132,False,/90kmxuSwU28dVy1ghVSHI4x1wb8.jpg,...,United States of America,English,"prostitute, native american, parody, anthology...","Ethan Coen, Joel Coen","Joel Coen, Ethan Coen, Jack London, Stewart Ed...",7.2,183199,"Tim Blake Nelson, Willie Watson, Clancy Brown,...","[Western, Comedy, Drama]","[Comedy, Drama]"
1200,569547,Black Mirror: Bandersnatch,6.768,3672,Released,2018-12-28,0,90,False,/1qSo3zT9XAQYcbZaLBn46mTeTHZ.jpg,...,"United States of America, United Kingdom",English,"video game, technology, choice, free will, bri...",David Slade,Charlie Brooker,7.1,142929,"Fionn Whitehead, Craig Parkinson, Alice Lowe, ...","[Science Fiction, Mystery, Drama, Thriller, TV...","[Drama, Thriller, TV Movie]"
1252,508965,Klaus,8.250,3542,Released,2019-11-08,0,96,False,/2u1YG0pgm5bIOXO2OVWLNdMl23f.jpg,...,Spain,"English, Northern Sami","friendship, small town, island, holiday, santa...","Sergio Pablos, Carlos Martínez López","Sergio Pablos, Jim Mahoney, Zach Lewis",8.2,234339,"Jason Schwartzman, J.K. Simmons, Rashida Jones...","[Animation, Family, Adventure, Comedy, Fantasy]","[Animation, Family, Comedy]"


In [8]:
mlb = MultiLabelBinarizer()
y_dataset = mlb.fit_transform(df['top_genres'].dropna())
genres_name = mlb.classes_
print(y_dataset)
print(genres_name)

[[0 0 1 ... 1 0 0]
 [0 0 1 ... 1 0 0]
 [0 0 1 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]]
['Action' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama' 'Family'
 'Horror' 'Music' 'Romance' 'TV Movie' 'Thriller']


In [9]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
X_vectors = embedder.encode(df['overview'].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9336 [00:00<?, ?it/s]

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vectors, 
    y_dataset, 
    test_size=0.2, 
    random_state=42
)

In [47]:
base_lr = LogisticRegression(max_iter=1000)
fast_model = MultiOutputClassifier(base_lr)
fast_model.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,LogisticRegre...max_iter=1000)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights invers

In [48]:
np.save('dataset/movie_vectors.npy', X_vectors)
joblib.dump(fast_model, "models/fast_genre_head.pkl")
np.save("models/genres_name.npy", genres_name)

In [53]:
y_pred_prob = fast_model.predict_proba(X_test)
y_pred = np.array([p[:, 1] > 0.5 for p in y_pred_prob]).T.astype(int)
print(classification_report(y_test, y_pred, target_names=genres_name))

              precision    recall  f1-score   support

      Action       0.67      0.27      0.39      4965
   Animation       0.74      0.38      0.50      4576
      Comedy       0.65      0.35      0.46     14559
       Crime       0.54      0.18      0.27      3876
 Documentary       0.79      0.65      0.71     10749
       Drama       0.66      0.57      0.61     23148
      Family       0.55      0.10      0.16      2871
      Horror       0.72      0.40      0.52      5169
       Music       0.71      0.41      0.52      2872
     Romance       0.57      0.17      0.26      5858
    TV Movie       0.58      0.01      0.02      3199
    Thriller       0.49      0.11      0.18      5167

   micro avg       0.68      0.39      0.50     87009
   macro avg       0.64      0.30      0.38     87009
weighted avg       0.66      0.39      0.47     87009
 samples avg       0.49      0.43      0.44     87009



/home/pai/git/movie-dss/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/pai/git/movie-dss/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/pai/git/movie-dss/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is"